# MODIS ET SSEBop - 1km
### 'projects/earthengine-legacy/assets/projects/usgs-ssebop/modis_et_v5_dekadal'

In [ ]:
import zipfile
from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/high_plains_quifer.zip'
extract_path = '/content/ogallala_shp'

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted to:", extract_path)

!pip install -q xee xarray netcdf4 geopandas pyproj
!pip install -q h5netcdf

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files extracted to: /content/ogallala_shp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.1 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 1. Install Required Packages & Setup Logging
# ==========================================
!pip install -q xee xarray netcdf4 geopandas geemap pyproj dask
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
gee_to_netcdf_v3.py — Earth Engine → CF-Compliant NetCDF Exporter
===================================================================

Changes from v2
---------------
FIX(16) scale_factor/add_offset written per-band as CF variable attributes
        so raw packed values can be decoded by any CF-compliant reader.
FIX(17) Per-band _FillValue: each variable gets its own native fill value.
        QA bitmask bands get uint16 dtype; angle bands get scale=0.01; etc.
FIX(18) grid_mapping variable (`spatial_ref`) added to every file.
        Contains full CRS WKT, GeoTransform, and CF parameters.
        Every data variable references it via grid_mapping='spatial_ref'.
FIX(19) Native integer dtypes preserved (int16 for NDVI, uint16 for QA).
        v2 cast everything to float32, losing packing + fill semantics.
FIX(20) Auto-detect band dtypes from GEE when BAND_METADATA is empty.
        Falls back to the detected dtype, logs warning about missing scale.

All v2 crash/logic/integrity fixes (FIX 1-15) are carried forward.
"""

# ╔════════════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION                                                     ║
# ╚════════════════════════════════════════════════════════════════════╝

DATASET_ID = "projects/earthengine-legacy/assets/projects/usgs-ssebop/modis_et_v5_dekadal"
PROJECT_ID = "msugw-503806"

SHAPEFILE_DIR = "/content/ogallala_shp"
ROI_LABEL     = "Ogallala"
DRIVE_ROOT    = "/content/drive/MyDrive/MSUGWB"

YEAR_START = None
YEAR_END   = None

DASK_WORKERS    = 4
CHUNK_XY        = 2048
COMPRESS_LEVEL  = 0
MAX_RETRIES     = 3
FORCE_OVERWRITE = False

# ── Per-Band Metadata (FIX 16/17/19) ────────────────────────────────
#
# Source: GEE Data Catalog for MODIS/061/MOD13A3
# https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13A3
#
# Keys per band:
#   dtype        – NetCDF storage type (int16, uint16, float32, …)
#   scale_factor – CF scale: physical = raw × scale + offset
#   add_offset   – CF offset (usually 0.0)
#   _FillValue   – raw value meaning "no data"
#   valid_range   – [min, max] of meaningful raw values (optional, CF attr)
#   long_name    – human-readable description (CF attr)
#   units        – physical units after decoding (CF attr)
#
# For a different dataset:
#   1. Change DATASET_ID
#   2. Replace BAND_METADATA with entries from the GEE catalog page
#   3. If you don't know the metadata, set BAND_METADATA = {}
#      → the pipeline will auto-detect dtypes and warn about missing scale
#

BAND_METADATA = {}


# ╔════════════════════════════════════════════════════════════════════╗
# ║  SETUP                                                             ║
# ╚════════════════════════════════════════════════════════════════════╝

# --- Colab prerequisites ---
# Uncomment the next two lines when running on Colab:
# !pip install -q "xee>=0.0.14" xarray netcdf4 geopandas pyproj
# from google.colab import drive; drive.mount('/content/drive')

import os, sys, glob, time, shutil, logging, gc, hashlib
from datetime import datetime, timezone

import numpy as np
import ee
import xarray as xr
import xee
from xee import helpers
import geopandas as gpd
import pyproj
import dask

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('gee_export')
logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)

dask.config.set(scheduler='threads', num_workers=DASK_WORKERS)

ee.Authenticate()
ee.Initialize(project=PROJECT_ID, opt_url='https://earthengine-highvolume.googleapis.com')
log.info('Earth Engine ready  (project=%s)', PROJECT_ID)


# ╔════════════════════════════════════════════════════════════════════╗
# ║  EXCEPTIONS                                                        ║
# ╚════════════════════════════════════════════════════════════════════╝

class DataQualityError(Exception):
    """Non-retryable data quality error."""
    pass


# ╔════════════════════════════════════════════════════════════════════╗
# ║  PIPELINE FUNCTIONS                                                ║
# ╚════════════════════════════════════════════════════════════════════╝

# ─── ROI ─────────────────────────────────────────────────────────────

def load_roi(shapefile_dir, simplify_deg=0.001):
    '''Load shapefile -> dissolved EPSG:4326 geometry.'''
    paths = glob.glob(os.path.join(shapefile_dir, '**', '*.shp'), recursive=True)
    if not paths:
        raise FileNotFoundError(f'No .shp found in {shapefile_dir}')
    log.info('Shapefile: %s', paths[0])

    gdf = gpd.read_file(paths[0])
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    roi = (gdf.geometry.union_all()
           if hasattr(gdf.geometry, 'union_all')
           else gdf.geometry.unary_union)

    try:
        import shapely as _shp
        nv = int(_shp.get_num_coordinates(roi))
    except Exception:
        nv = len(roi.wkt) // 20

    if nv > 50_000:
        roi = roi.simplify(simplify_deg, preserve_topology=True)
        try:
            nv_after = int(_shp.get_num_coordinates(roi))
        except Exception:
            nv_after = len(roi.wkt) // 20
        log.warning('Simplified ROI: %s -> %s vertices (tol=%.4f deg)',
                    f'{nv:,}', f'{nv_after:,}', simplify_deg)
    else:
        log.info('ROI vertices: %s', f'{nv:,}')
    return roi


# ─── CRS & Grid ─────────────────────────────────────────────────────

def resolve_native_crs(collection):
    '''Detect native CRS + nominal scale.  Returns (epsg_str, metres).'''
    proj = collection.first().select(0).projection()
    info = proj.getInfo()
    native_crs = info.get('crs', 'EPSG:4326')
    nominal_m  = proj.nominalScale().getInfo()

    log.info('Native CRS    : %s', native_crs[:70])
    log.info('Nominal scale : %.1f m', nominal_m)

    resolved = 'EPSG:4326'

    if native_crs.startswith('SR-ORG:'):
        log.warning('SR-ORG CRS (%s) -> EPSG:4326 fallback', native_crs)
    elif native_crs.startswith('EPSG:'):
        resolved = native_crs
    else:
        try:
            crs_obj = pyproj.CRS.from_user_input(native_crs)
            epsg = crs_obj.to_epsg()
            if epsg:
                resolved = f'EPSG:{epsg}'
            elif crs_obj.axis_info and crs_obj.axis_info[0].unit_name == 'metre':
                resolved = 'EPSG:5070'
                log.warning('Metre CRS, no EPSG -> EPSG:5070')
        except Exception as exc:
            log.warning('CRS parse error: %s -> EPSG:4326', exc)

    log.info('Resolved CRS  : %s', resolved)
    return resolved, nominal_m


def build_grid_params(roi, grid_crs, nominal_m):
    '''Compute crs_transform + shape_2d for the actual ROI subset.

    Returns the dict that XEE's open_dataset() expects.
    The crs_transform describes THIS subset, not the global grid (FIX 18).
    '''
    minx, miny, maxx, maxy = roi.bounds

    transformer = pyproj.Transformer.from_crs(
        'EPSG:4326', grid_crs, always_xy=True)
    x1, y1 = transformer.transform(minx, miny)
    x2, y2 = transformer.transform(maxx, maxy)

    left   = min(x1, x2)
    right  = max(x1, x2)
    bottom = min(y1, y2)
    top    = max(y1, y2)

    try:
        unit = pyproj.CRS.from_user_input(grid_crs).axis_info[0].unit_name
    except Exception:
        unit = 'degree'

    if unit == 'metre':
        scale_x = float(nominal_m)
        scale_y = float(nominal_m)
    else:
        lat_centroid = roi.centroid.y
        cos_lat = np.cos(np.radians(lat_centroid))
        scale_y = nominal_m / 111_320.0
        scale_x = scale_y / cos_lat if cos_lat > 0.01 else scale_y

    width  = int(np.ceil((right - left) / scale_x))
    height = int(np.ceil((top - bottom) / scale_y))

    crs_transform = (scale_x, 0.0, left, 0.0, -scale_y, top)

    log.info('Grid : %d x %d px  |  scale_x=%.6g  scale_y=%.6g %s',
             width, height, scale_x, scale_y, unit)

    return {
        'crs': grid_crs,
        'crs_transform': crs_transform,
        'shape_2d': (width, height),
    }


# ─── Band Metadata Detection (FIX 20) ───────────────────────────────

def detect_band_types(collection, bands):
    '''Auto-detect native dtypes from GEE ee.Image.bandTypes().

    Returns {band_name: {'dtype': 'int16'|'uint16'|'float32'|...}}.
    '''
    detected = {}
    try:
        img = collection.first()
        bt = img.bandTypes().getInfo()
        for band_name in bands:
            if band_name not in bt:
                continue
            info = bt[band_name]
            precision = info.get('precision', 'float')
            mn = info.get('min', 0)
            mx = info.get('max', 0)

            if precision == 'int':
                if mn >= 0 and mx <= 255:
                    dtype = 'uint8'
                elif mn >= 0 and mx <= 65535:
                    dtype = 'uint16'
                elif mn >= -32768 and mx <= 32767:
                    dtype = 'int16'
                else:
                    dtype = 'int32'
            elif precision == 'double':
                dtype = 'float64'
            else:
                dtype = 'float32'

            detected[band_name] = {'dtype': dtype}
            log.info('  Band %-20s : %s (GEE: %s, range [%s, %s])',
                     band_name, dtype, precision, mn, mx)
    except Exception as e:
        log.warning('Cannot auto-detect band types: %s', e)

    return detected


def merge_band_info(bands, user_metadata, auto_detected):
    '''Merge user-provided BAND_METADATA with auto-detected types.

    User metadata takes precedence.  Auto-detected provides dtype fallback.
    Bands with no metadata at all get float32 + warning.
    '''
    result = {}
    warned = False

    for band in bands:
        info = {}

        # Layer 1: auto-detected dtype
        if band in auto_detected:
            info.update(auto_detected[band])

        # Layer 2: user-provided overrides everything
        if band in user_metadata:
            info.update(user_metadata[band])

        # Ensure dtype exists
        if 'dtype' not in info:
            info['dtype'] = 'float32'

        # Default _FillValue based on dtype if not user-specified
        if '_FillValue' not in info:
            dt = np.dtype(info['dtype'])
            if np.issubdtype(dt, np.unsignedinteger):
                info['_FillValue'] = int(np.iinfo(dt).max)       # e.g. 65535
            elif np.issubdtype(dt, np.signedinteger):
                info['_FillValue'] = int(np.iinfo(dt).min)       # e.g. -32768
            else:
                info['_FillValue'] = None   # NaN for floats (handled by encoding)

        # Warn about missing scale_factor (only once)
        if 'scale_factor' not in info and not warned:
            if band not in user_metadata:
                log.warning(
                    'BAND_METADATA missing for "%s" (and possibly others). '
                    'Raw values will be written without scale_factor/add_offset. '
                    'Populate BAND_METADATA from the GEE catalog for proper CF decoding.',
                    band)
                warned = True

        result[band] = info

    return result


# ─── Encoding & Metadata ────────────────────────────────────────────

def strip_xee_encoding(ds):
    '''Clear ALL XEE encoding (the bogus CRS-derived scale_factor).

    GEE already returns raw pixel values via computePixels.  XEE injects
    the CRS pixel size as scale_factor, which is NOT a CF data-packing
    parameter.  Clearing it prevents xarray from mis-interpreting it.
    '''
    for name in list(ds.data_vars) + list(ds.coords):
        if name in ds:
            ds[name].encoding.clear()
    return ds


def apply_band_metadata(ds, band_info):
    '''Write scale_factor, add_offset, valid_range, units, long_name
    as VARIABLE ATTRIBUTES — not encoding.

    FIX(16): scale_factor/add_offset are CF variable attributes.
    xarray's encoding dict interprets them as packing instructions
    (it would divide data by scale_factor before writing).  Since our
    data is ALREADY packed as raw integers from GEE, we must NOT put
    scale/offset in the encoding.  Writing them as attrs means:
      - xarray writes the raw values as-is
      - scale_factor appears in the file as a variable attribute
      - any CF reader (xarray, CDO, ncview) decodes automatically

    FIX(17): Per-band _FillValue is set individually.
    '''
    for var in list(ds.data_vars):
        if var not in band_info:
            continue
        info = band_info[var]

        # CF scale/offset as attributes
        if 'scale_factor' in info:
            ds[var].attrs['scale_factor'] = np.float64(info['scale_factor'])
        if 'add_offset' in info:
            ds[var].attrs['add_offset'] = np.float64(info['add_offset'])

        # Optional CF attributes
        if 'valid_range' in info:
            ds[var].attrs['valid_range'] = np.array(
                info['valid_range'], dtype=info.get('dtype', 'float32'))
        if 'long_name' in info:
            ds[var].attrs['long_name'] = info['long_name']
        if 'units' in info:
            ds[var].attrs['units'] = info['units']
        if 'flag_meanings' in info:
            ds[var].attrs['flag_meanings'] = info['flag_meanings']

    return ds


def prepare_for_write(ds, band_info):
    '''Cast variables to their target dtype, filling NaN → _FillValue.

    FIX(19): preserves native integer dtypes instead of blanket float32.
    For int16/uint16 bands, NaN (a float concept) is replaced with the
    band's _FillValue before the cast to integer.

    All operations are lazy (dask) — no eager computation.
    '''
    for var in list(ds.data_vars):
        if var not in band_info:
            continue

        info = band_info[var]
        target_dtype = np.dtype(info['dtype'])
        fill_val = info.get('_FillValue')

        if np.issubdtype(target_dtype, np.integer):
            # Integer dtypes cannot represent NaN.
            # Replace NaN with the fill value, then cast.
            if fill_val is None:
                fill_val = int(np.iinfo(target_dtype).min)
            ds[var] = ds[var].fillna(fill_val).astype(target_dtype)
        else:
            # Float dtypes: NaN stays as NaN (natural missing indicator)
            ds[var] = ds[var].astype(target_dtype)

    return ds


def build_encoding(ds, band_info, compress_level=4, chunk_xy=512):
    '''Build NetCDF encoding dict with per-band dtype and _FillValue.

    FIX(3):  chunksizes capped to actual dims.
    FIX(17): per-band _FillValue.
    FIX(19): native dtype from band_info.

    IMPORTANT: scale_factor and add_offset are NOT in the encoding.
    They are variable attributes set by apply_band_metadata().
    '''
    enc = {}

    for var in ds.data_vars:
        info = band_info.get(var, {})
        dtype = info.get('dtype', 'float32')
        fill = info.get('_FillValue')

        # For float types with no explicit fill, use NaN
        if fill is None and np.issubdtype(np.dtype(dtype), np.floating):
            fill = np.float32(np.nan) if dtype == 'float32' else np.float64(np.nan)

        # Cast fill to the target dtype for HDF5 compatibility
        if fill is not None:
            try:
                fill = np.dtype(dtype).type(fill)
            except (ValueError, OverflowError):
                pass

        # Chunk sizes capped to actual dimensions
        var_chunks = []
        for dim in ds[var].dims:
            dim_size = ds[var].sizes[dim]
            if dim == 'time':
                var_chunks.append(min(1, dim_size))
            elif dim in ('x', 'y'):
                var_chunks.append(min(chunk_xy, dim_size))
            else:
                var_chunks.append(min(1, dim_size))

        enc[var] = {
            'dtype': dtype,
            '_FillValue': fill,
            'zlib': compress_level > 0,
            'complevel': compress_level,
            'chunksizes': tuple(var_chunks),
        }

    # Coordinate encoding
    for c in ds.coords:
        if c == 'time':
            enc[c] = {
                'dtype': 'int64',
                '_FillValue': None,
                'units': 'days since 1970-01-01',
                'calendar': 'proleptic_gregorian',
            }
        elif c in ('x', 'y'):
            enc[c] = {'dtype': 'float64', '_FillValue': None}

    return enc


# ─── CF Grid Mapping (FIX 18) ───────────────────────────────────────

def add_grid_mapping(ds, grid_crs, crs_transform, grid_shape):
    '''Add a CF-compliant grid_mapping variable (spatial_ref).

    FIX(18): The file previously had no grid_mapping despite declaring
    CF-1.8, and the CRS metadata described the global MODIS grid, not
    the actual ROI subset.

    This function creates a scalar `spatial_ref` variable with:
      - grid_mapping_name + all CF projection parameters (via pyproj)
      - Full CRS WKT for precision
      - GeoTransform describing THIS subset (not the global grid)
      - Actual grid dimensions
    Every data variable gets a grid_mapping='spatial_ref' attribute.
    '''
    crs_obj = pyproj.CRS.from_user_input(grid_crs)

    # pyproj.CRS.to_cf() returns all required CF parameters:
    #   grid_mapping_name, semi_major_axis, inverse_flattening, etc.
    cf_attrs = crs_obj.to_cf()

    # Add full WKT for tools that prefer it (GDAL, QGIS, rioxarray)
    wkt = crs_obj.to_wkt()
    cf_attrs['crs_wkt']     = wkt
    cf_attrs['spatial_ref'] = wkt   # GDAL convention

    # GeoTransform for THIS subset (GDAL convention):
    #   (x_origin, x_pixel_size, x_rotation, y_origin, y_rotation, y_pixel_size)
    # Our crs_transform: (x_scale, 0, x_origin, 0, -y_scale, y_origin)
    geo_transform = (
        f'{crs_transform[2]} {crs_transform[0]} {crs_transform[1]} '
        f'{crs_transform[5]} {crs_transform[3]} {crs_transform[4]}'
    )
    cf_attrs['GeoTransform'] = geo_transform

    # Grid dimensions of the actual subset
    width, height = grid_shape
    cf_attrs['grid_width']  = int(width)
    cf_attrs['grid_height'] = int(height)

    # Create the scalar grid_mapping variable
    ds['spatial_ref'] = xr.DataArray(
        data=np.int32(0),
        attrs=cf_attrs,
    )

    # Tag every data variable with grid_mapping
    for var in list(ds.data_vars):
        if var != 'spatial_ref':
            ds[var].attrs['grid_mapping'] = 'spatial_ref'

    return ds


def make_cf_compliant(ds, dataset_id, roi_label, grid_crs,
                      crs_transform, grid_shape, year):
    '''Stamp CF-1.8 global + coordinate attributes + grid_mapping.

    FIX(18): now includes grid_mapping variable and subset-specific
    GeoTransform instead of global grid metadata.
    '''
    width, height = grid_shape

    ds.attrs.update({
        'Conventions': 'CF-1.8',
        'title':  f'{dataset_id} -- {roi_label} ({year})',
        'source': f'Google Earth Engine: {dataset_id}',
        'history': f'Created {datetime.now(timezone.utc).isoformat()}',
        'crs': grid_crs,
        'geospatial_bounds_crs': grid_crs,
        'grid_width': int(width),
        'grid_height': int(height),
    })

    if 'time' in ds.coords:
        ds['time'].attrs.update(axis='T', standard_name='time')

    try:
        is_proj = (pyproj.CRS.from_user_input(grid_crs)
                   .axis_info[0].unit_name == 'metre')
    except Exception:
        is_proj = False

    if 'x' in ds.coords:
        ds['x'].attrs.update(
            axis='X',
            standard_name='projection_x_coordinate' if is_proj else 'longitude',
            units='m' if is_proj else 'degrees_east',
        )
    if 'y' in ds.coords:
        ds['y'].attrs.update(
            axis='Y',
            standard_name='projection_y_coordinate' if is_proj else 'latitude',
            units='m' if is_proj else 'degrees_north',
        )

    # Add grid_mapping variable
    ds = add_grid_mapping(ds, grid_crs, crs_transform, grid_shape)

    return ds


def sanitize_attrs(ds):
    '''Flatten complex GEE metadata; drop None values.'''
    for var in list(ds.data_vars) + list(ds.coords):
        if var not in ds:
            continue
        to_drop = []
        for key, val in ds[var].attrs.items():
            if val is None:
                to_drop.append(key)
            elif not isinstance(val, (str, int, float, np.number, np.ndarray)):
                ds[var].attrs[key] = str(val)
        for key in to_drop:
            del ds[var].attrs[key]

    to_drop = []
    for key, val in ds.attrs.items():
        if val is None:
            to_drop.append(key)
        elif not isinstance(val, (str, int, float, np.number, np.ndarray)):
            ds.attrs[key] = str(val)
    for key in to_drop:
        del ds.attrs[key]

    return ds


# ─── Pre-Write Validation ──────────────────────────────────────────

# def pre_write_check(ds, year):
#     '''Coarsened subsample across the entire grid, ALL variables.'''
#     for var_name in ds.data_vars:
#         if var_name == 'spatial_ref':
#             continue  # skip the grid_mapping scalar
#         da = ds[var_name]

#         coarsen_dims = {}
#         for d in ('x', 'y'):
#             if d in da.dims and da.sizes[d] > 1:
#                 coarsen_dims[d] = max(1, da.sizes[d] // 50)

#         if not coarsen_dims:
#             log.warning('[%d] %s: no spatial dims to check', year, var_name)
#             continue

#         indexers = {'time': 0} if 'time' in da.dims else {}

#         # Use .max() for int bands (no NaN propagation), check for
#         # the fill value instead of np.isfinite for integers
#         sample = (da.isel(**indexers)
#                     .coarsen(coarsen_dims, boundary='trim')
#                     .max()
#                     .compute())

#         vals = sample.values

#         if np.issubdtype(vals.dtype, np.floating):
#             nf = int(np.isfinite(vals).sum())
#         else:
#             # Integer: count non-fill values
#             fill = da.attrs.get('_FillValue',
#                                 da.encoding.get('_FillValue', None))
#             if fill is not None:
#                 nf = int((vals != fill).sum())
#             else:
#                 nf = int(vals.size)   # no fill → assume all valid

#         if nf == 0:
#             raise DataQualityError(
#                 f'[{year}] Variable "{var_name}" has no valid pixels.')
#         log.info('[%d] Pre-check %s: %d valid values OK', year, var_name, nf)

# ─── Pre-Write Validation ──────────────────────────────────────────

def pre_write_check(ds, year):
    '''Coarsened subsample across the entire grid, checking ONLY the first band.

    Checking all bands triggers duplicate network downloads from GEE.
    If the CRS/Geometry is broken, the first band will be NaN, which tells
    us everything we need to know without wasting time.
    '''
    # Find the first actual data variable (skip spatial_ref)
    data_vars = [v for v in ds.data_vars if v != 'spatial_ref']
    if not data_vars:
        return  # No data to check

    var_name = data_vars[0]
    da = ds[var_name]

    coarsen_dims = {}
    for d in ('x', 'y'):
        if d in da.dims and da.sizes[d] > 1:
            coarsen_dims[d] = max(1, da.sizes[d] // 50)

    if not coarsen_dims:
        log.warning('[%d] %s: no spatial dims to check', year, var_name)
        return

    indexers = {'time': 0} if 'time' in da.dims else {}

    # Use .max() for int bands (no NaN propagation), check for
    # the fill value instead of np.isfinite for integers
    sample = (da.isel(**indexers)
                .coarsen(coarsen_dims, boundary='trim')
                .max()
                .compute())

    vals = sample.values

    if np.issubdtype(vals.dtype, np.floating):
        nf = int(np.isfinite(vals).sum())
    else:
        # Integer: count non-fill values
        fill = da.attrs.get('_FillValue',
                            da.encoding.get('_FillValue', None))
        if fill is not None:
            nf = int((vals != fill).sum())
        else:
            nf = int(vals.size)   # no fill → assume all valid

    if nf == 0:
        raise DataQualityError(
            f'[{year}] Variable "{var_name}" has no valid pixels. '
            'Check GEE asset availability, ROI, and CRS.')

    log.info('[%d] Pre-check OK: %d valid values found in %s', year, nf, var_name)

# ─── Post-Write Integrity Verification ─────────────────────────────

def post_write_verify(filepath, expected_vars, band_info, year):
    '''Reopen the NetCDF and verify structure + data integrity.

    FIX(11): checks ALL bands.
    FIX(16): now expects legitimate scale_factor on bands that declare it
             in BAND_METADATA (no longer flags them as "bogus").
    '''
    try:
        vds = xr.open_dataset(filepath, chunks='auto', mask_and_scale=False)
    except Exception as e:
        return False, f'Cannot reopen: {e}'

    try:
        missing = [v for v in expected_vars if v not in vds.data_vars]
        if missing:
            return False, f'Missing vars: {missing}'

        if 'time' not in vds.dims:
            return False, 'No time dimension'

        # grid_mapping present?
        if 'spatial_ref' not in vds:
            return False, 'Missing spatial_ref grid_mapping variable'

        # Coordinates finite?
        for c in ('x', 'y'):
            if c in vds.coords:
                vals = vds[c].values
                if np.issubdtype(vals.dtype, np.floating):
                    if not np.all(np.isfinite(vals)):
                        return False, f'Non-finite coord: {c}'

        # Check scale_factor: should match BAND_METADATA (not CRS pixel size)
        for v in expected_vars:
            info = band_info.get(v, {})
            expected_sf = info.get('scale_factor')
            actual_sf = vds[v].attrs.get('scale_factor',
                                         vds[v].encoding.get('scale_factor'))

            if expected_sf is not None:
                if actual_sf is None:
                    return False, f'{v}: expected scale_factor={expected_sf}, got none'
                if abs(float(actual_sf) - float(expected_sf)) > 1e-10:
                    return False, (
                        f'{v}: scale_factor={actual_sf} != expected {expected_sf} '
                        '(possible XEE pollution)')
            else:
                # Band with no expected scale — should have none
                if actual_sf is not None and float(actual_sf) != 1.0:
                    return False, (
                        f'{v}: unexpected scale_factor={actual_sf}')

        # Data has valid values? (sample ALL variables)
        bad_vars = []
        for v in expected_vars:
            da = vds[v]
            info = band_info.get(v, {})
            fill_val = info.get('_FillValue')

            idx = {}
            if 'time' in da.dims:
                idx['time'] = 0
            for d in da.dims:
                if d != 'time':
                    s = da.sizes[d]
                    m = s // 2
                    hw = max(1, min(50, s // 4))
                    idx[d] = slice(max(0, m - hw), min(s, m + hw))

            samp = da.isel(**idx).compute().values

            if np.issubdtype(samp.dtype, np.floating):
                valid = np.any(np.isfinite(samp))
            else:
                if fill_val is not None:
                    valid = np.any(samp != fill_val)
                else:
                    valid = samp.size > 0

            if not valid:
                bad_vars.append(v)

        if bad_vars:
            return False, f'No valid data on re-read: {bad_vars}'

        return True, f'All {len(expected_vars)} vars verified with correct metadata'

    finally:
        vds.close()


# ─── Drive Copy ─────────────────────────────────────────────────────

def get_md5(filepath, chunk_bytes=10 * 1024 * 1024):
    '''MD5 with standard buffered I/O (no O_DIRECT — FUSE incompatible).'''
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    '''Copy to Drive, flush, verify MD5.'''
    src_md5 = get_md5(src)

    for attempt in range(1, max_retries + 1):
        try:
            log.info('Drive copy attempt %d/%d ...', attempt, max_retries)

            with open(src, 'rb') as f_src, open(dst, 'wb') as f_dst:
                shutil.copyfileobj(f_src, f_dst, length=10 * 1024 * 1024)
                f_dst.flush()
                os.fsync(f_dst.fileno())

            os.sync()
            time.sleep(3)

            src_size = os.path.getsize(src)
            dst_size = os.path.getsize(dst)
            if src_size != dst_size:
                raise IOError(
                    f'Size mismatch: src={src_size}, dst={dst_size}')

            dst_md5 = get_md5(dst)
            if src_md5 != dst_md5:
                raise IOError('MD5 mismatch')

            log.info('Drive copy verified (MD5 match, %d bytes)', src_size)
            return True

        except Exception as e:
            log.error('Drive copy attempt %d failed: %s', attempt, e)
            if os.path.exists(dst):
                try:
                    os.remove(dst)
                except OSError:
                    pass
            if attempt < max_retries:
                wait = 15 * attempt
                log.info('Retrying in %ds ...', wait)
                time.sleep(wait)
            else:
                raise DataQualityError(
                    f'Drive copy failed after {max_retries} attempts: {e}')


# ╔════════════════════════════════════════════════════════════════════╗
# ║  MAIN PIPELINE                                                     ║
# ╚════════════════════════════════════════════════════════════════════╝

def run_pipeline():
    log.info('=' * 62)
    log.info('  Dataset : %s', DATASET_ID)
    log.info('=' * 62)

    # ── 1. Collection metadata ──────────────────────────────────
    coll = ee.ImageCollection(DATASET_ID)
    bands = coll.first().bandNames().getInfo()
    if not bands:
        raise ValueError(f'{DATASET_ID} returned no bands')

    try:
        y0 = YEAR_START or int(ee.Date(
            coll.sort('system:time_start').first()
                .get('system:time_start')).get('year').getInfo())
        y1 = YEAR_END or int(ee.Date(
            coll.sort('system:time_start', False).first()
                .get('system:time_start')).get('year').getInfo())
    except Exception as exc:
        raise ValueError(
            f'Cannot determine year range: {exc}') from exc

    years = list(range(y0, y1 + 1))
    log.info('Bands (%d) : %s', len(bands), bands)
    log.info('Years      : %d - %d  (%d total)', y0, y1, len(years))

    # ── 2. CRS & grid ──────────────────────────────────────────
    grid_crs, nominal_m = resolve_native_crs(coll.select(bands))

    tolerance_deg = (nominal_m * 1.5) / 111_320.0
    roi = load_roi(SHAPEFILE_DIR, simplify_deg=tolerance_deg)

    short = "openET_ensemble_ET"
    out_dir = os.path.join(DRIVE_ROOT, short)
    os.makedirs(out_dir, exist_ok=True)
    log.info('Output : %s', out_dir)

    grid_params = build_grid_params(roi, grid_crs, nominal_m)
    crs_transform = grid_params['crs_transform']
    grid_shape    = grid_params['shape_2d']

    # ── 3. Band metadata (auto-detect + user override) ──────────
    log.info('Detecting band types from GEE ...')
    auto_types = detect_band_types(coll.select(bands), bands)
    band_info  = merge_band_info(bands, BAND_METADATA, auto_types)

    log.info('Band info summary:')
    for b in bands:
        info = band_info[b]
        sf = info.get('scale_factor', '—')
        fv = info.get('_FillValue', '—')
        log.info('  %-20s  dtype=%-7s  scale=%-8s  fill=%s',
                 b, info['dtype'], sf, fv)

    # ── 4. Year loop ────────────────────────────────────────────
    n_ok = n_fail = n_skip = 0

    for year in years:
        fname = f'{short}_{year}_{ROI_LABEL}.nc'
        final = os.path.join(out_dir, fname)

        # ── Resume check ────────────────────────────────────────
        if os.path.exists(final) and not FORCE_OVERWRITE:
            sz = os.path.getsize(final)
            if sz < 1024:
                log.warning('[%d] Tiny file (%d B) — re-processing', year, sz)
                os.remove(final)
            else:
                is_valid = False
                try:
                    with xr.open_dataset(final, mask_and_scale=False) as chk:
                        if ('time' in chk.dims and chk.sizes['time'] > 0
                                and 'spatial_ref' in chk):
                            is_valid = True
                except Exception:
                    pass

                if is_valid:
                    log.info('[%d] On Drive (%.1f MiB), valid — skip',
                             year, sz / 1024**2)
                    n_skip += 1
                    continue
                else:
                    log.warning('[%d] Existing file missing metadata — re-processing', year)
                    try:
                        os.remove(final)
                    except OSError:
                        pass

        tmp = os.path.join('/content', fname)
        done = False

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                log.info('[%d] attempt %d/%d', year, attempt, MAX_RETRIES)

                yr_col = (
                    ee.ImageCollection(DATASET_ID)
                    .filter(ee.Filter.calendarRange(year, year, 'year'))
                    .select(bands)
                    .sort('system:time_start')
                )

                n_img = yr_col.size().getInfo()
                if n_img == 0:
                    log.warning('[%d] 0 images — skip year', year)
                    n_skip += 1
                    break

                log.info('[%d] %d image(s)', year, n_img)

                # ── Open lazily ──────────────────────────────────
                grid_w, grid_h = grid_shape
                safe_chunk = max(16, min(CHUNK_XY, grid_w, grid_h))

                ds = xr.open_dataset(
                    yr_col,
                    engine='ee',
                    chunks={'x': safe_chunk, 'y': safe_chunk},
                    **grid_params,
                )

                # Deduplicate timestamps
                _, unique_idx = np.unique(ds['time'], return_index=True)
                if len(unique_idx) != ds.sizes['time']:
                    log.warning('[%d] Deduplicating timestamps', year)
                    ds = ds.isel(time=sorted(unique_idx))

                # ── Fix encoding ─────────────────────────────────
                ds = strip_xee_encoding(ds)

                # ── Apply band metadata (scale, fill, units) ─────
                ds = apply_band_metadata(ds, band_info)

                # ── Cast to native dtypes (NaN → fill for ints) ──
                ds = prepare_for_write(ds, band_info)

                # ── CF compliance + grid_mapping ─────────────────
                ds = sanitize_attrs(ds)
                ds = make_cf_compliant(
                    ds, DATASET_ID, ROI_LABEL, grid_crs,
                    crs_transform, grid_shape, year)

                # ── Pre-write validation ─────────────────────────
                pre_write_check(ds, year)

                # ── Write to scratch ─────────────────────────────
                enc = build_encoding(ds, band_info, COMPRESS_LEVEL, safe_chunk)

                log.info('[%d] Writing -> %s', year, tmp)
                t0 = time.time()

                try:
                    with dask.config.set(scheduler='synchronous'):
                        ds.to_netcdf(
                            tmp,
                            engine='netcdf4',
                            encoding=enc,
                            unlimited_dims=['time'],
                        )
                except Exception:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                    raise

                dt = time.time() - t0
                mb = os.path.getsize(tmp) / 1024**2
                log.info('[%d] Written in %.0fs  (%.1f MiB)', year, dt, mb)

                if os.path.getsize(tmp) < 1024:
                    raise DataQualityError(
                        f'[{year}] File too small ({os.path.getsize(tmp)} B)')

                # ── Post-write integrity ─────────────────────────
                ok_flag, msg = post_write_verify(tmp, bands, band_info, year)
                if not ok_flag:
                    raise DataQualityError(
                        f'[{year}] INTEGRITY FAIL: {msg}')
                log.info('[%d] Integrity OK: %s', year, msg)

                # ── Export to Drive ───────────────────────────────
                log.info('[%d] Exporting to Drive ...', year)
                t_mv = time.time()
                robust_drive_copy(tmp, final, max_retries=3)
                os.remove(tmp)
                log.info('[%d] Exported in %.0fs -> %s',
                         year, time.time() - t_mv, final)

                n_ok += 1
                done = True
                break

            except DataQualityError as e:
                log.error('[%d] DATA ERROR (not retrying): %s', year, e)
                n_fail += 1
                break

            except Exception as e:
                log.error('[%d] attempt %d error: %s', year, attempt, e)
                if attempt < MAX_RETRIES:
                    wait = 30 * 2 ** (attempt - 1)
                    log.info('[%d] retry in %ds ...', year, wait)
                    time.sleep(wait)
                else:
                    log.error('[%d] retries exhausted', year)
                    n_fail += 1

            finally:
                if not done and os.path.exists(tmp):
                    try:
                        os.remove(tmp)
                    except OSError:
                        pass
                gc.collect()
                gc.collect()

    # ── Summary ─────────────────────────────────────────────────
    log.info('=' * 62)
    log.info('  DONE  %d ok | %d fail | %d skip  (%d years)',
             n_ok, n_fail, n_skip, len(years))
    log.info('=' * 62)

    if os.path.isdir(out_dir):
        nc_files = sorted(f for f in os.listdir(out_dir) if f.endswith('.nc'))
        if nc_files:
            log.info('Files on Drive (%s):', out_dir)
            for f in nc_files:
                sz = os.path.getsize(os.path.join(out_dir, f)) / 1024**2
                log.info('  %s  (%.1f MiB)', f, sz)


if __name__ == '__main__':
    run_pipeline()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 66.3 MB/s eta 0:00:00


MessageError: Error: credential propagation was unsuccessful

# OpenET Ensemble Monthly Evapotranspiration v2.1

### ee.ImageCollection("projects/openet/assets/ensemble/conus/gridmet/monthly/v2_1")

In [ ]:
# ==========================================
# 1. Install Required Packages
# ==========================================
!pip install -q xee xarray netcdf4 geopandas geemap pyproj dask
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
gee_to_netcdf_v5_5.py — Earth Engine → CF-Compliant NetCDF Exporter
OpenET 30 m, TILED + CHECKPOINTED edition

Changes from v5.4
-----------------
FIX(53)  MONTHLY MOSAIC: the v2_1 asset stores each month as ~32 HUC2
         watershed SHARD images (system:index '11t_20200601_20200630'),
         each masked outside its basin.  That single fact produced every
         prior failure: pixel probes hit shards not covering the test
         point, mask probes saw median-of-mostly-masked, and XEE saw
         ~32 duplicate timesteps/month (-> dedup kept arbitrary shards
         -> all-NaN tiles).  We now group by system:time_start and
         mosaic each month into ONE CONUS image before any processing.
FIX(54)  The 6 catalog bands ARE the data product (et_ensemble_mad =
         ensemble mean after MAD outlier filtering — NOT an ancillary
         stat).  SELECT_BANDS = all 6, EXCLUDE_BANDS = [], BAND_METADATA
         written from the catalog descriptions.
FIX(55)  Native CRS/transform read from the RAW shard images: mosaics
         report EE's default 1-degree projection, which would have
         wrecked the grid (111 km pixels).  Mosaic only feeds pixels.
"""

# ════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ════════════════════════════════════════════════════════════════════

DATASET_ID = "projects/openet/assets/ensemble/conus/gridmet/monthly/v2_1"
PROJECT_ID = "msugw-503806"

SHAPEFILE_DIR = "/content/ogallala_shp"
ROI_LABEL     = "Ogallala"
DRIVE_ROOT    = "/content/drive/MyDrive/MSUGWB"
SCRATCH_DIR   = "/content/openet_scratch"

YEAR_START = None
YEAR_END   = None

# FIX(54): the 6 catalog bands = the complete data product.
SELECT_BANDS = None        # None = all 6 bands below
EXCLUDE_BANDS = []         # nothing excluded — all 6 bands carry data

PIXEL_PROBE_ENFORCE = True # fail fast if a probe finds no real pixels

TILING      = (4, 4)
BLOCK_YEARS = 1

DISK_MIN_FREE_GB      = 20.0
DISK_CHECK_INTERVAL_S = 600
DISK_MAX_WAIT_S       = 6 * 3600
AUTO_CLEANUP          = True

CHECKPOINT_SECONDS    = 3600
CHECKPOINT_MIN_NEW_GB = 2.0
KEEP_CHECKPOINTS      = False

PROBE_LATEST_N       = 12
MASK_PROBE_ENFORCE   = False

DROP_EMPTY_BANDS       = True
CAST_DOUBLE_TO_FLOAT32 = True

DASK_WORKERS    = 8
WRITE_SCHEDULER = 'threads'
PROGRESS_LOG_S  = 120

CHUNK_XY        = 1024
COMPRESS_LEVEL  = 4
MAX_RETRIES     = 4
FORCE_OVERWRITE = False

# FIX(54): descriptions straight from the catalog page.
BAND_METADATA = {
    'et_ensemble_mad':       {'dtype': 'float32', 'units': 'mm',
        'long_name': 'Ensemble monthly ET (mean of ensemble after MAD outlier filtering)'},
    'et_ensemble_mad_min':   {'dtype': 'float32', 'units': 'mm',
        'long_name': 'Minimum of ensemble monthly ET range after MAD outlier filtering'},
    'et_ensemble_mad_max':   {'dtype': 'float32', 'units': 'mm',
        'long_name': 'Maximum of ensemble monthly ET range after MAD outlier filtering'},
    'et_ensemble_mad_count': {'dtype': 'uint8', 'units': '1',
        'long_name': 'Number of models used in ensemble ET after MAD outlier filtering'},
    'et_ensemble_mad_index': {'dtype': 'uint16', 'units': '1',
        'long_name': 'Bitmask of models included in the ensemble ET (MAD filtering)'},
    'et_ensemble_sam':       {'dtype': 'float32', 'units': 'mm',
        'long_name': 'Simple arithmetic mean (SAM) of all six OpenET models, monthly ET'},
}

# ════════════════════════════════════════════════════════════════════
#  SETUP
# ════════════════════════════════════════════════════════════════════

# from google.colab import drive; drive.mount('/content/drive')

import os, re, glob, time, math, shutil, logging, gc, hashlib, threading
import subprocess, random, difflib
from datetime import datetime, timezone

import numpy as np
import ee
import xarray as xr
import xee
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import dask
from dask.diagnostics import Callback

for _n in ('google_auth_httplib2', 'googleapiclient.discovery_cache',
           'googleapiclient.http', 'urllib3.connectionpool'):
    logging.getLogger(_n).setLevel(logging.ERROR)

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)-7s | %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('gee_export')

dask.config.set(scheduler='threads', num_workers=DASK_WORKERS)
os.makedirs(SCRATCH_DIR, exist_ok=True)

ee.Authenticate()
ee.Initialize(project=PROJECT_ID,
              opt_url='https://earthengine-highvolume.googleapis.com')
log.info('Earth Engine ready (project=%s, high-volume endpoint)', PROJECT_ID)


class DataQualityError(Exception):
    pass


class ProgressCallback(Callback):
    def __init__(self, label, interval_s=PROGRESS_LOG_S):
        super().__init__()
        self.label, self.iv = label, interval_s

    def _start(self, dsk):
        self.t0, self.n_tot, self.n, self._last = time.time(), len(dsk), 0, 0.0
        log.info('[%s] write started: %d chunks total', self.label,
                 self.n_tot)

    def _posttask(self, key, result, dsk, state, worker_id):
        self.n += 1
        now = time.time()
        if now - self._last >= self.iv:
            self._last = now
            log.info('[%s] write %d/%d chunks (%.0f s elapsed)',
                     self.label, self.n, self.n_tot, now - self.t0)


# ─── Band name resolution ───────────────────────────────────────────

def resolve_select_bands(requested, available, fuzzy_cutoff=0.7):
    def norm(s):
        return re.sub(r'[^a-z0-9]', '', s.lower())
    norm_avail = {norm(b): b for b in available}
    mapping, unmatched = {}, []
    for req in requested:
        if req in available:
            mapping[req] = req
            continue
        nr = norm(req)
        if nr in norm_avail:
            mapping[req] = norm_avail[nr]
            continue
        substr = [b for b in available
                  if nr and (nr in norm(b) or norm(b) in nr)]
        if substr:
            mapping[req] = sorted(substr, key=len)[0]
            continue
        close = difflib.get_close_matches(nr, list(norm_avail), n=1,
                                          cutoff=fuzzy_cutoff)
        if close:
            mapping[req] = norm_avail[close[0]]
        else:
            unmatched.append(req)
    resolved = sorted(set(mapping.values()), key=available.index)
    return resolved, mapping, unmatched


# ─── FIX(53): HUC-shard -> monthly mosaic ───────────────────────────

def monthly_mosaic(col):
    '''Group shard images by system:time_start and mosaic each month.
    Shards are spatially disjoint HUC2 basins -> mosaic = clean union,
    one CONUS image per month, chronological order preserved.'''
    def _mos(d):
        d = ee.Number(d)
        return (col.filter(ee.Filter.eq('system:time_start', d))
                   .mosaic()
                   .set('system:time_start', d)
                   .set('system:index', ee.Date(d).format('YYYYMM')))
    dates = ee.List(col.aggregate_array('system:time_start')
                        .distinct()).sort()
    return ee.ImageCollection(dates.map(_mos))


def dedup_by_time(col):
    '''Distinct timestamps, re-wrapped (Collection.distinct returns a
    FeatureCollection-typed object that crashes XEE).'''
    return ee.ImageCollection(col.distinct('system:time_start'))


# ─── ROI / CRS / grid / tiling ──────────────────────────────────────

def load_roi(shapefile_dir, simplify_deg=0.001):
    paths = glob.glob(os.path.join(shapefile_dir, '**', '*.shp'),
                      recursive=True)
    if not paths:
        raise FileNotFoundError(f'No .shp found in {shapefile_dir}')
    gdf = gpd.read_file(paths[0])
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    roi = (gdf.geometry.union_all() if hasattr(gdf.geometry, 'union_all')
           else gdf.geometry.unary_union)
    return roi.simplify(simplify_deg, preserve_topology=True)


def resolve_native_crs(collection):
    '''FIX(55): call this on RAW shard images, never on a mosaic
    (composites report EE's default 1-degree projection).'''
    proj = collection.first().select(0).projection()
    info = proj.getInfo()
    native_crs = info.get('crs', 'EPSG:4326')
    nominal_m  = proj.nominalScale().getInfo()
    transform  = info.get('transform')
    log.info('Native CRS    : %s', native_crs[:70])
    log.info('Nominal scale : %.2f m', nominal_m)
    resolved = 'EPSG:4326'
    if native_crs.startswith('EPSG:'):
        resolved = native_crs
    elif native_crs.startswith('SR-ORG:'):
        log.warning('SR-ORG CRS (%s) -> EPSG:4326 fallback', native_crs)
    else:
        try:
            c = pyproj.CRS.from_user_input(native_crs)
            e = c.to_epsg()
            if e:
                resolved = f'EPSG:{e}'
            elif c.axis_info and c.axis_info[0].unit_name == 'metre':
                resolved = 'EPSG:5070'
        except Exception as exc:
            log.warning('CRS parse error: %s -> EPSG:4326', exc)
    log.info('Resolved CRS  : %s', resolved)
    return resolved, nominal_m, transform


def _snap(v, step, origin, side):
    k = (v - origin) / step
    if abs(k - round(k)) < 1e-6:
        k = int(round(k))
    else:
        k = math.floor(k) if side == 'down' else math.ceil(k)
    return origin + k * step


def build_grid_params(roi, grid_crs, nominal_m, native_transform=None):
    minx, miny, maxx, maxy = roi.bounds
    tr = pyproj.Transformer.from_crs('EPSG:4326', grid_crs, always_xy=True)
    xs_t, ys_t = tr.transform([minx, maxx], [miny, maxy])
    left, right = min(xs_t), max(xs_t)
    bottom, top = min(ys_t), max(ys_t)
    try:
        unit = pyproj.CRS.from_user_input(grid_crs).axis_info[0].unit_name
    except Exception:
        unit = 'degree'
    if native_transform and len(native_transform) == 6 \
            and float(native_transform[0]):
        sx = abs(float(native_transform[0]))
        sy = abs(float(native_transform[4]))
        x0, y0 = float(native_transform[2]), float(native_transform[5])
        left  = _snap(left,  sx, x0, 'down')
        right = _snap(right, sx, x0, 'up')
        top    = y0 - math.floor((y0 - top) / sy) * sy
        bottom = y0 - math.ceil((y0 - bottom) / sy) * sy
        scale_x, scale_y = sx, sy
        log.info('Snapped to native grid (step=%.8g)', sx)
    elif unit == 'metre':
        scale_x = scale_y = float(nominal_m)
    else:
        scale_x = scale_y = float(nominal_m) / 111_319.4908
    width  = max(1, int(round((right - left) / scale_x)))
    height = max(1, int(round((top - bottom) / scale_y)))
    log.info('ROI grid : %d x %d px  (%.2f GPx)', width, height,
             width * height / 1e9)
    return {'crs': grid_crs,
            'crs_transform': (scale_x, 0.0, left, 0.0, -scale_y, top),
            'shape_2d': (width, height)}


def split_tiles(grid_params, tiling=(2, 2)):
    W, H = grid_params['shape_2d']
    a, _b, x0, _d, e, y0 = grid_params['crs_transform']
    sx, sy = a, -e
    ty, tx = tiling
    xe = [round(W * i / tx) for i in range(tx + 1)]
    ye = [round(H * i / ty) for i in range(ty + 1)]
    tiles, idx = [], 0
    for r in range(ty):
        for c in range(tx):
            c0, c1 = xe[c], xe[c + 1]
            r0, r1 = ye[r], ye[r + 1]
            tiles.append({
                'idx': idx, 'row': r, 'col': c, 'label': f'tile{idx}',
                'width': c1 - c0, 'height': r1 - r0,
                'crs_transform': (sx, 0.0, x0 + c0 * sx, 0.0, -sy,
                                  y0 - r0 * sy),
                'shape_2d': (c1 - c0, r1 - r0),
                'px_offset': (c0, r0)})
            idx += 1
    log.info('Tile plan (%d tiles):', len(tiles))
    for t in tiles:
        log.info('  %-6s  %5d x %5d px  (offset x=%d y=%d)',
                 t['label'], t['width'], t['height'], *t['px_offset'])
    return tiles


def tile_bounds_4326(tile, grid_crs):
    a, _b, x0, _d, e, y0 = tile['crs_transform']
    w, h = tile['shape_2d']
    inv = pyproj.Transformer.from_crs(grid_crs, 'EPSG:4326', always_xy=True)
    lon = sorted(inv.transform([x0, x0 + a * w], [y0 + e * h, y0])[0])
    lat = sorted(inv.transform([x0, x0 + a * w], [y0 + e * h, y0])[1])
    return float(lon[0]), float(lat[0]), float(lon[-1]), float(lat[-1])


# ─── Band metadata ──────────────────────────────────────────────────

def detect_band_types(collection, bands):
    detected = {}
    try:
        bt = collection.first().bandTypes().getInfo()
        for b in bands:
            if b not in bt:
                continue
            info = bt[b]
            prec = info.get('precision', 'float')
            mn, mx = info.get('min', 0), info.get('max', 0)
            if prec == 'int':
                if mn >= 0 and mx <= 255:          dt = 'uint8'
                elif mn >= 0 and mx <= 65535:      dt = 'uint16'
                elif mn >= -32768 and mx <= 32767: dt = 'int16'
                else:                              dt = 'int32'
            elif prec == 'double':
                dt = 'float64'
            else:
                dt = 'float32'
            detected[b] = {'dtype': dt}
    except Exception as e:
        log.warning('Cannot auto-detect band types: %s', e)
    return detected


def merge_band_info(bands, user_metadata, auto_detected):
    result, warned = {}, False
    for band in bands:
        info = {}
        info.update(auto_detected.get(band, {}))
        info.update(user_metadata.get(band, {}))
        if 'dtype' not in info:
            info['dtype'] = 'float32'
        if CAST_DOUBLE_TO_FLOAT32 and info['dtype'] == 'float64':
            info['dtype'] = 'float32'
        if '_FillValue' not in info:
            dt = np.dtype(info['dtype'])
            if np.issubdtype(dt, np.unsignedinteger):
                info['_FillValue'] = int(np.iinfo(dt).max)
            elif np.issubdtype(dt, np.signedinteger):
                info['_FillValue'] = int(np.iinfo(dt).min)
            else:
                info['_FillValue'] = None
        if 'scale_factor' not in info and not warned \
                and band not in user_metadata:
            log.warning('BAND_METADATA missing for "%s" — written without '
                        'scale_factor (OK for OpenET: values in mm).', band)
            warned = True
        result[band] = info
    return result


# ─── Encoding & CF metadata ─────────────────────────────────────────

def strip_xee_encoding(ds):
    for name in list(ds.data_vars) + list(ds.coords):
        if name in ds:
            ds[name].encoding.clear()
    return ds


def apply_band_metadata(ds, band_info):
    for var in list(ds.data_vars):
        if var not in band_info:
            continue
        info = band_info[var]
        if 'scale_factor' in info:
            ds[var].attrs['scale_factor'] = np.float64(info['scale_factor'])
        if 'add_offset' in info:
            ds[var].attrs['add_offset'] = np.float64(info['add_offset'])
        if 'valid_range' in info:
            ds[var].attrs['valid_range'] = np.array(
                info['valid_range'], dtype=info.get('dtype', 'float32'))
        for key in ('long_name', 'units', 'flag_meanings'):
            if key in info:
                ds[var].attrs[key] = info[key]
    return ds


def prepare_for_write(ds, band_info):
    for var in list(ds.data_vars):
        if var not in band_info:
            continue
        info = band_info[var]
        tdt = np.dtype(info['dtype'])
        fv = info.get('_FillValue')
        if np.issubdtype(tdt, np.integer):
            if fv is None:
                fv = int(np.iinfo(tdt).min)
            ds[var] = ds[var].fillna(fv).astype(tdt)
        else:
            ds[var] = ds[var].astype(tdt)
    return ds


def build_encoding(ds, band_info, compress_level=4, chunk_xy=512):
    enc = {}
    for var in ds.data_vars:
        info = band_info.get(var, {})
        dtype = info.get('dtype', 'float32')
        fill = info.get('_FillValue')
        if fill is None and np.issubdtype(np.dtype(dtype), np.floating):
            fill = (np.float32(np.nan) if dtype == 'float32'
                    else np.float64(np.nan))
        if fill is not None:
            try:
                fill = np.dtype(dtype).type(fill)
            except (ValueError, OverflowError):
                pass
        ch = []
        for dim in ds[var].dims:
            s = ds[var].sizes[dim]
            if dim == 'time':
                ch.append(min(1, s))
            elif dim in ('x', 'y'):
                ch.append(min(chunk_xy, s))
            else:
                ch.append(min(1, s))
        enc[var] = {'dtype': dtype, '_FillValue': fill,
                    'zlib': compress_level > 0, 'complevel': compress_level,
                    'shuffle': True, 'chunksizes': tuple(ch)}
    for c in ds.coords:
        if c == 'time':
            enc[c] = {'dtype': 'int64', '_FillValue': None,
                      'units': 'days since 1970-01-01',
                      'calendar': 'proleptic_gregorian'}
        elif c in ('x', 'y'):
            enc[c] = {'dtype': 'float64', '_FillValue': None}
    return enc


def add_grid_mapping(ds, grid_crs, crs_transform, grid_shape):
    crs_obj = pyproj.CRS.from_user_input(grid_crs)
    cf = crs_obj.to_cf()
    wkt = crs_obj.to_wkt()
    cf['crs_wkt'] = wkt
    cf['spatial_ref'] = wkt
    cf['GeoTransform'] = (f'{crs_transform[2]} {crs_transform[0]} '
                          f'{crs_transform[1]} {crs_transform[5]} '
                          f'{crs_transform[3]} {crs_transform[4]}')
    w, h = grid_shape
    cf['grid_width'], cf['grid_height'] = int(w), int(h)
    ds['spatial_ref'] = xr.DataArray(data=np.int32(0), attrs=cf)
    for var in list(ds.data_vars):
        if var != 'spatial_ref':
            ds[var].attrs['grid_mapping'] = 'spatial_ref'
    return ds


def make_cf_compliant(ds, dataset_id, roi_label, grid_crs,
                      crs_transform, grid_shape, period_label, tile=None):
    w, h = grid_shape
    ds.attrs.update({
        'Conventions': 'CF-1.8',
        'title': f'{dataset_id} -- {roi_label} '
                 f'{tile["label"] if tile else ""} ({period_label})',
        'source': f'Google Earth Engine: {dataset_id}',
        'history': f'Created {datetime.now(timezone.utc).isoformat()}',
        'crs': grid_crs,
        'geospatial_bounds_crs': grid_crs,
        'grid_width': int(w), 'grid_height': int(h)})
    if tile is not None:
        ds.attrs.update({
            'tile_index': int(tile['idx']),
            'tile_row': int(tile['row']), 'tile_col': int(tile['col']),
            'tile_x_pixel_offset': int(tile['px_offset'][0]),
            'tile_y_pixel_offset': int(tile['px_offset'][1])})
    if 'time' in ds.coords:
        ds['time'].attrs.update(axis='T', standard_name='time')
    try:
        is_proj = (pyproj.CRS.from_user_input(grid_crs)
                   .axis_info[0].unit_name == 'metre')
    except Exception:
        is_proj = False
    if 'x' in ds.coords:
        ds['x'].attrs.update(
            axis='X',
            standard_name=('projection_x_coordinate' if is_proj
                           else 'longitude'),
            units='m' if is_proj else 'degrees_east')
    if 'y' in ds.coords:
        ds['y'].attrs.update(
            axis='Y',
            standard_name=('projection_y_coordinate' if is_proj
                           else 'latitude'),
            units='m' if is_proj else 'degrees_north')
    return add_grid_mapping(ds, grid_crs, crs_transform, grid_shape)


def sanitize_attrs(ds):
    for var in list(ds.data_vars) + list(ds.coords):
        if var not in ds:
            continue
        drop = []
        for k, v in ds[var].attrs.items():
            if v is None:
                drop.append(k)
            elif not isinstance(v, (str, int, float, np.number, np.ndarray)):
                ds[var].attrs[k] = str(v)
        for k in drop:
            del ds[var].attrs[k]
    drop = []
    for k, v in ds.attrs.items():
        if v is None:
            drop.append(k)
        elif not isinstance(v, (str, int, float, np.number, np.ndarray)):
            ds.attrs[k] = str(v)
    for k in drop:
        del ds.attrs[k]
    return ds


# ─── Dataset validation (fail fast) ─────────────────────────────────

def scan_recent(coll, n=5):
    fc = ee.FeatureCollection(
        coll.sort('system:time_start', False).limit(n)
            .map(lambda im: ee.Feature(None, {
                'i': ee.String(im.get('system:index')),
                'b': im.bandNames()})))
    return [(f['properties']['i'], list(f['properties']['b']))
            for f in fc.getInfo()['features']]


def pixel_probe_ok(coll, bands, roi):
    rp = roi.representative_point()
    pt = ee.Geometry.Point([float(rp.x), float(rp.y)])
    imgs = (coll.sort('system:time_start', False)
                .select(bands).limit(3).toList(3))
    for k in range(3):
        try:
            vals = ee.Image(imgs.get(k)).reduceRegion(
                ee.Reducer.first(), pt, 90).getInfo()
            if vals and any(v is not None for v in vals.values()):
                log.info('Pixel probe OK on image %d: %s', k,
                         {kk: vv for kk, vv in vals.items()
                          if vv is not None})
                return True
        except Exception as e:
            log.warning('Pixel probe image %d failed: %s', k, e)
    return False


def validate_dataset(coll, roi, select=None):
    inv = scan_recent(coll, n=5)
    if not inv:
        raise ValueError(f'{DATASET_ID} returned no images')
    for idx, bnames in inv:
        log.info('  newest image %-24s #bands=%d', idx, len(bnames))

    union = []
    for _, bnames in inv:
        for n in bnames:
            if n not in union:
                union.append(n)
    usable = [b for b in union if b not in set(EXCLUDE_BANDS)]
    excluded = [b for b in union if b in set(EXCLUDE_BANDS)]
    if excluded:
        log.warning('EXCLUDE_BANDS dropped band(s): %s', excluded)

    if select:
        resolved, mapping, unmatched = resolve_select_bands(select, usable)
        for req, act in mapping.items():
            if act != req:
                log.warning('Band remap: "%s" -> "%s"', req, act)
        if unmatched:
            log.warning('SELECT_BANDS not found: %s', unmatched)
        if not resolved:
            raise ValueError(
                f'None of SELECT_BANDS {select} exist in {DATASET_ID}.\n'
                f'Available bands: {usable}')
        usable = resolved

    if not usable:
        raise ValueError(f'{DATASET_ID} has no usable bands '
                         f'(available: {union})')

    if PIXEL_PROBE_ENFORCE:
        if not pixel_probe_ok(coll, usable, roi):
            raise DataQualityError(
                f'Pixel probe FAILED on {DATASET_ID} for bands {usable} '
                f'at the ROI centroid (on the monthly MOSAIC).  Aborting '
                f'before any download.')
    return usable


# ─── Per-tile mask probe (memory-safe) ──────────────────────────────

def source_empty_bands(col, bands, bbox_4326, scale_m):
    def _run(geom):
        probe = (col.sort('system:time_start', False)
                    .limit(PROBE_LATEST_N).median())
        return (probe.mask().reduceRegion(
                    reducer=ee.Reducer.anyNonZero(), geometry=geom,
                    scale=max(scale_m * 4, 100), bestEffort=True,
                    maxPixels=1e9, tileScale=8).getInfo())
    try:
        res = _run(ee.Geometry.Rectangle(list(bbox_4326)))
    except Exception as exc:
        log.warning('Mask probe full-tile failed (%s) — retrying inner '
                    'quarter', exc)
        try:
            minx, miny, maxx, maxy = bbox_4326
            cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
            inner = [cx - (maxx - minx) / 4, cy - (maxy - miny) / 4,
                     cx + (maxx - minx) / 4, cy + (maxy - miny) / 4]
            res = _run(ee.Geometry.Rectangle(inner))
        except Exception as exc2:
            log.warning('Mask probe failed (%s) — assuming all bands valid',
                        exc2)
            return set()
    empty = {b for b in bands if not res.get(b)}
    if empty:
        log.warning('Bands fully masked at source over tile: %s',
                    sorted(empty))
    return empty


# ─── netCDF4 block merger ───────────────────────────────────────────

def merge_block_into_master(master_path, block_path, label):
    create = not os.path.exists(master_path)
    mst = nc4.Dataset(master_path, 'w' if create else 'a')
    try:
        with nc4.Dataset(block_path, 'r') as blk:
            nt_blk = blk.dimensions['time'].size
            if create:
                mst.setncatts({k: blk.getncattr(k) for k in blk.ncattrs()})
                mst.createDimension('time', None)
                for d in ('y', 'x'):
                    mst.createDimension(d, blk.dimensions[d].size)
                    bv = blk.variables[d]
                    v = mst.createVariable(d, bv.dtype, (d,))
                    v[:] = bv[:]
                    v.setncatts({k: bv.getncattr(k) for k in bv.ncattrs()})
                if 'spatial_ref' in blk.variables:
                    bv = blk.variables['spatial_ref']
                    v = mst.createVariable('spatial_ref', bv.dtype, ())
                    v[...] = bv[...]
                    v.setncatts({k: bv.getncattr(k) for k in bv.ncattrs()})
            if 'time' not in mst.variables:
                bt = blk.variables['time']
                v = mst.createVariable('time', bt.dtype, ('time',))
                v.setncatts({k: bt.getncattr(k) for k in bt.ncattrs()})
            t_off = mst.dimensions['time'].size
            mst.variables['time'][t_off:t_off + nt_blk] = \
                blk.variables['time'][:]
            for name, bv in blk.variables.items():
                if name in ('time', 'y', 'x', 'spatial_ref'):
                    continue
                if name not in mst.variables:
                    encf = bv.filters() or {}
                    chunks = bv.chunking()
                    fill = (bv.getncattr('_FillValue')
                            if '_FillValue' in bv.ncattrs() else None)
                    v = mst.createVariable(
                        name, bv.dtype, bv.dimensions,
                        zlib=bool(encf.get('zlib')),
                        complevel=int(encf.get('complevel', 4)),
                        shuffle=bool(encf.get('shuffle')),
                        fill_value=fill,
                        chunksizes=tuple(chunks) if chunks else None)
                    v.setncatts({k: bv.getncattr(k)
                                 for k in bv.ncattrs()
                                 if k != '_FillValue'})
                mst.variables[name][t_off:t_off + nt_blk, ...] = bv[:]
        mst.sync()
    finally:
        mst.close()
    log.info('[%s] merged %d timesteps (master now %d)',
             label, nt_blk, t_off + nt_blk)


# ─── Validation helpers ─────────────────────────────────────────────

def pre_write_check(ds, label, n_samples=25, block_size=100):
    data_vars = [v for v in ds.data_vars if v != 'spatial_ref']
    if not data_vars:
        return
    da = ds[data_vars[0]]
    nx, ny = da.sizes.get('x', 0), da.sizes.get('y', 0)
    if nx == 0 or ny == 0:
        return
    bs = min(block_size, nx, ny)
    for i in range(n_samples):
        sx = random.randint(0, nx - bs)
        sy = random.randint(0, ny - bs)
        idx = {'x': slice(sx, sx + bs), 'y': slice(sy, sy + bs)}
        if 'time' in da.dims:
            idx['time'] = 0
        vals = da.isel(**idx).compute().values
        if np.issubdtype(vals.dtype, np.floating):
            if np.isfinite(vals).sum() > 0:
                log.info('[%s] Pre-check OK (%d random blocks)', label,
                         i + 1)
                return
        else:
            log.info('[%s] Pre-check OK (%d random blocks)', label, i + 1)
            return
    raise DataQualityError(
        f'[{label}] Monte Carlo pre-check failed: no valid pixels in ROI.')


def post_write_verify(filepath, expected_vars, band_info, label):
    try:
        vds = xr.open_dataset(filepath, chunks='auto', mask_and_scale=False)
    except Exception as e:
        return False, f'Cannot reopen: {e}'
    try:
        missing = [v for v in expected_vars if v not in vds.data_vars]
        if missing:
            return False, f'Missing vars: {missing}'
        if 'time' not in vds.dims or vds.sizes['time'] == 0:
            return False, 'No time dimension / empty'
        if 'spatial_ref' not in vds:
            return False, 'Missing spatial_ref'
        for v in expected_vars:
            sf = vds[v].attrs.get('scale_factor',
                                  vds[v].encoding.get('scale_factor'))
            if sf is not None and float(sf) != 1.0:
                return False, f'{v}: unexpected scale_factor={sf}'
        nt = vds.sizes['time']
        ts = (sorted(set([0, nt // 4, nt // 2, (3 * nt) // 4, nt - 1]))
              if nt > 3 else list(range(nt)))
        bad = []
        for v in expected_vars:
            da, info = vds[v], band_info.get(v, {})
            fv = info.get('_FillValue')
            found = False
            for t in ts:
                idx = {'time': t} if 'time' in da.dims else {}
                for d in da.dims:
                    if d == 'time':
                        continue
                    s = da.sizes[d]
                    m, hw = s // 2, max(1, min(50, s // 4))
                    idx[d] = slice(max(0, m - hw), min(s, m + hw))
                samp = da.isel(**idx).compute().values
                if np.issubdtype(samp.dtype, np.floating):
                    if np.any(np.isfinite(samp)):
                        found = True
                        break
                elif fv is not None:
                    if np.any(samp != fv):
                        found = True
                        break
                else:
                    found = True
                    break
            if not found:
                bad.append(v)
        if len(bad) == len(expected_vars):
            return False, f'No valid data for ANY band: {bad}'
        if bad:
            log.warning('[%s] Sparse bands tolerated: %s', label, bad)
        return True, (f'{len(expected_vars) - len(bad)}'
                      f'/{len(expected_vars)} vars verified; {nt} timesteps')
    finally:
        vds.close()


# ─── Disk watchdog ──────────────────────────────────────────────────

def free_gb(path=SCRATCH_DIR):
    try:
        return shutil.disk_usage(path).free / 1024**3
    except Exception:
        return 1e9


def cleanup_scratch():
    freed = free_gb()
    for pat in ('*.part', '*.tmp', '*.lock'):
        for f in glob.glob(os.path.join(SCRATCH_DIR, pat)):
            try:
                os.remove(f)
            except OSError:
                pass
    if AUTO_CLEANUP:
        try:
            subprocess.run(['pip', 'cache', 'purge'],
                           capture_output=True, timeout=120)
        except Exception:
            pass
    return free_gb() - freed


def ensure_disk_free(context='', extra_gb=0.0):
    need = DISK_MIN_FREE_GB + max(0.0, extra_gb)
    deadline = time.time() + DISK_MAX_WAIT_S
    first = True
    while True:
        f = free_gb()
        if f >= need:
            if not first:
                log.info('Disk OK: %.1f GB free (need %.1f)', f, need)
            return
        if first:
            log.warning('LOW DISK [%.1f < %.1f GB] %s — cleaning & waiting',
                        f, need, context)
            first = False
        cleanup_scratch()
        if free_gb() >= need:
            continue
        if time.time() > deadline:
            raise DataQualityError(
                f'Disk never recovered ({free_gb():.1f}/{need:.1f} GB) '
                f'{context} — aborting (resumable).')
        time.sleep(60)


class DiskWatchdog(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True, name='disk-watchdog')
        self._stop = threading.Event()

    def run(self):
        beat = 0
        while not self._stop.is_set():
            f = free_gb()
            if f < DISK_MIN_FREE_GB:
                log.warning('WATCHDOG: low disk %.1f < %.1f GB',
                            f, DISK_MIN_FREE_GB)
                cleanup_scratch()
            else:
                beat += 1
                if beat % 6 == 1:
                    log.info('WATCHDOG: disk healthy (%.1f GB free)', f)
            self._stop.wait(DISK_CHECK_INTERVAL_S)


# ─── Drive copy / checkpoints ───────────────────────────────────────

def get_md5(filepath, chunk_bytes=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        while chunk := f.read(chunk_bytes):
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, verify='md5', max_retries=3):
    src_size = os.path.getsize(src)
    src_md5 = get_md5(src) if verify == 'md5' else None
    for attempt in range(1, max_retries + 1):
        try:
            with open(src, 'rb') as fs, open(dst, 'wb') as fd:
                shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                fd.flush()
                os.fsync(fd.fileno())
            os.sync()
            time.sleep(3)
            if os.path.getsize(dst) != src_size:
                raise IOError(f'Size mismatch {os.path.getsize(dst)}')
            if verify == 'md5' and get_md5(dst) != src_md5:
                raise IOError('MD5 mismatch')
            log.info('Drive copy verified (%s, %.1f MiB)',
                     verify, src_size / 1024**2)
            return True
        except Exception as e:
            log.error('Drive copy attempt %d failed: %s', attempt, e)
            if os.path.exists(dst):
                try:
                    os.remove(dst)
                except OSError:
                    pass
            if attempt < max_retries:
                time.sleep(15 * attempt)
            else:
                raise DataQualityError(f'Drive copy failed: {e}')


def valid_nc(path):
    try:
        with nc4.Dataset(path, 'r') as d:
            return ('time' in d.variables and len(d.variables['time']) > 0
                    and 'spatial_ref' in d.variables)
    except Exception:
        return False


def read_last_time(path):
    try:
        with nc4.Dataset(path, 'r') as d:
            if 'time' not in d.variables:
                return None
            t = d.variables['time']
            if len(t) == 0:
                return None
            dtv = nc4.num2date(float(t[-1]), t.units,
                               getattr(t, 'calendar', 'standard'),
                               only_use_cftime_datetimes=False,
                               only_use_python_datetimes=True)
            return np.datetime64(dtv.replace(tzinfo=None), 'ns')
    except Exception as e:
        log.warning('read_last_time failed on %s: %s', path, e)
        return None


def find_latest_checkpoint(drive_dir, tile_idx, base):
    ck = re.compile(rf'^{re.escape(base)}tile({tile_idx})_(\d+)hr\.nc$')
    best, best_hr = None, -1
    if not os.path.isdir(drive_dir):
        return None
    for fname in os.listdir(drive_dir):
        m = ck.match(fname)
        if m and int(m.group(2)) > best_hr:
            best, best_hr = os.path.join(drive_dir, fname), int(m.group(2))
    return best


# ─── Metadata-first month listing / block prep ──────────────────────

def list_months(col_block):
    ts = col_block.aggregate_array('system:time_start').getInfo()
    return np.unique(np.array(ts, dtype='datetime64[ms]')) \
        .astype('datetime64[ns]')


def prepare_block_dataset(col_block, tile, grid_crs, band_info, label,
                          chunk):
    ds = xr.open_dataset(
        col_block, engine='ee',
        chunks={'x': chunk, 'y': chunk},
        crs=grid_crs,
        crs_transform=tile['crs_transform'],
        shape_2d=tile['shape_2d'])
    _, uniq = np.unique(ds['time'].values, return_index=True)
    if len(uniq) != ds.sizes['time']:
        log.warning('[%s] XEE returned duplicate times — fixing', label)
        ds = ds.isel(time=sorted(uniq))
    ds = strip_xee_encoding(ds)
    ds = apply_band_metadata(ds, band_info)
    ds = prepare_for_write(ds, band_info)
    ds = sanitize_attrs(ds)
    ds = make_cf_compliant(ds, DATASET_ID, ROI_LABEL, grid_crs,
                           tile['crs_transform'], tile['shape_2d'],
                           f'{label} block', tile=tile)
    return ds


# ════════════════════════════════════════════════════════════════════
#  TILE PROCESSING
# ════════════════════════════════════════════════════════════════════

def process_tile(tile, grid_crs, nominal_m, band_info, years, out_dir,
                 base):
    label = tile['label']
    local = os.path.join(SCRATCH_DIR, f'{base}{label}.nc')
    final = os.path.join(out_dir, f'{base}{label}.nc')

    if os.path.exists(final) and not FORCE_OVERWRITE and valid_nc(final):
        log.info('[%s] Final file on Drive (%.1f MiB) — skip',
                 label, os.path.getsize(final) / 1024**2)
        return 'skip'

    ckpt = find_latest_checkpoint(out_dir, tile['idx'], base)
    if not os.path.exists(local) and ckpt and not FORCE_OVERWRITE:
        log.info('[%s] Restoring checkpoint %s (%.1f MiB) ...', label,
                 os.path.basename(ckpt),
                 os.path.getsize(ckpt) / 1024**2)
        ensure_disk_free(f'{label} checkpoint restore')
        shutil.copy2(ckpt, local)
    if os.path.exists(local) and not valid_nc(local):
        log.warning('[%s] Local file invalid — starting fresh', label)
        try:
            os.remove(local)
        except OSError:
            pass
    last_time = read_last_time(local) if os.path.exists(local) else None
    if last_time is not None:
        log.info('[%s] Resuming after %s', label, str(last_time)[:16])

    bbox = tile_bounds_4326(tile, grid_crs)

    # FIX(53): mosaic the HUC shards into one image per month
    col_full = monthly_mosaic(
        ee.ImageCollection(DATASET_ID)
          .filterDate(f'{years[0]}-01-01', f'{years[-1]+1}-01-01'))

    probe_bands = list(band_info.keys())
    empty = source_empty_bands(col_full, probe_bands, bbox, nominal_m)

    select_bands = list(probe_bands)
    if empty and DROP_EMPTY_BANDS and len(empty) < len(probe_bands):
        select_bands = [b for b in probe_bands if b not in empty]
        log.warning('[%s] Dropping %d masked band(s): %s', label,
                    len(empty), sorted(empty))
    if len(empty) == len(probe_bands) and MASK_PROBE_ENFORCE:
        log.error('[%s] probe says ALL bands masked and '
                  'MASK_PROBE_ENFORCE=True — skipping', label)
        return 'fail'

    col_tile = dedup_by_time(col_full.select(select_bands))

    months_all = list_months(col_tile)
    log.info('[%s] %d unique months in %d-%d (%d bands, %d x %d px)',
             label, len(months_all), years[0], years[-1],
             len(select_bands), tile['width'], tile['height'])

    chunk = max(16, min(CHUNK_XY, tile['width'], tile['height']))
    blocks = [years[i:i + BLOCK_YEARS]
              for i in range(0, len(years), BLOCK_YEARS)]
    next_ckpt = time.time() + CHECKPOINT_SECONDS
    ckpt_base_bytes = os.path.getsize(local) if os.path.exists(local) else 0

    for byears in blocks:
        blk_px = tile['width'] * tile['height']
        blk_gb = (len(byears) * 12 * blk_px * len(select_bands)
                  * 4 * 0.55 / 1024**3)
        ensure_disk_free(f'{label} block {byears[0]}', extra_gb=blk_gb * 0.5)

        lo = np.datetime64(f'{byears[0]}-01-01', 'ns')
        hi = np.datetime64(f'{byears[-1]+1}-01-01', 'ns')
        blk_months = months_all[(months_all >= lo) & (months_all < hi)]
        if last_time is not None:
            blk_months = blk_months[blk_months > last_time]

        if len(blk_months) == 0:
            log.info('[%s] block %d-%d: no new months (checked BEFORE '
                     'opening XEE)', label, byears[0], byears[-1])
            continue

        col_block = dedup_by_time(
            col_tile.filterDate(str(blk_months[0])[:10],
                                str(blk_months[-1])[:10] + 'T00:00:00'))
        log.info('[%s] block %d-%d: %d month(s) to fetch: %s .. %s',
                 label, byears[0], byears[-1], len(blk_months),
                 str(blk_months[0])[:7], str(blk_months[-1])[:7])

        done = False
        for attempt in range(1, MAX_RETRIES + 1):
            block_path = None
            try:
                ds = prepare_block_dataset(col_block, tile, grid_crs,
                                           band_info, label, chunk)
                if ds.sizes.get('time', 0) == 0:
                    log.info('[%s] block %d-%d: nothing after resume cut',
                             label, byears[0], byears[-1])
                    ds.close()
                    done = True
                    break

                if not os.path.exists(local):
                    pre_write_check(ds, label)

                enc = build_encoding(ds, band_info, COMPRESS_LEVEL, chunk)
                block_path = os.path.join(
                    SCRATCH_DIR, f'{base}{label}.y{byears[0]}.nc')
                t0 = time.time()
                log.info('[%s] block %d-%d -> scratch (parallel fetch, '
                         'progress every %ds) ...', label, byears[0],
                         byears[-1], PROGRESS_LOG_S)
                try:
                    with dask.config.set(scheduler=WRITE_SCHEDULER), \
                            ProgressCallback(label):
                        ds.to_netcdf(block_path, engine='netcdf4',
                                     encoding=enc, mode='w')
                finally:
                    del ds
                    gc.collect()

                ensure_disk_free(
                    f'{label} merge block {byears[0]}',
                    extra_gb=os.path.getsize(block_path) / 1024**3)
                merge_block_into_master(local, block_path, label)
                log.info('[%s] block %d-%d done in %.0fs '
                         '(%.1f MiB total)', label, byears[0], byears[-1],
                         time.time() - t0,
                         os.path.getsize(local) / 1024**2)
                done = True
                break

            except DataQualityError:
                raise
            except Exception as e:
                log.error('[%s] block error (attempt %d): %s',
                          label, attempt, e)
                if attempt < MAX_RETRIES:
                    time.sleep(min(300, 30 * 2 ** (attempt - 1)))
            finally:
                if block_path and os.path.exists(block_path):
                    try:
                        os.remove(block_path)
                    except OSError:
                        pass
                gc.collect()

        if not done:
            log.error('[%s] block %d-%d failed after %d attempts — '
                      'tile RESUMABLE at last checkpoint',
                      label, byears[0], byears[-1], MAX_RETRIES)
            return 'incomplete'

        last_time = read_last_time(local)

        if time.time() >= next_ckpt:
            grown = os.path.getsize(local) - ckpt_base_bytes
            if grown >= CHECKPOINT_MIN_NEW_GB * 1024**3:
                hr = int((time.time() - _run_start) // 3600) or 1
                dst = os.path.join(out_dir, f'{base}{label}_{hr}hr.nc')
                ensure_disk_free(f'{label} checkpoint copy')
                log.info('[%s] Hourly checkpoint -> %s (+%.1f GB)', label,
                         os.path.basename(dst), grown / 1024**3)
                try:
                    robust_drive_copy(local, dst, verify='size')
                    ckpt_base_bytes = os.path.getsize(local)
                except Exception as e:
                    log.error('[%s] checkpoint copy failed: %s', label, e)
            else:
                log.info('[%s] checkpoint skipped (%.2f GB new)', label,
                         grown / 1024**3)
            next_ckpt = time.time() + CHECKPOINT_SECONDS

    if not os.path.exists(local):
        log.error('[%s] nothing written — tile failed', label)
        return 'fail'

    ensure_disk_free(f'{label} final verify+export',
                     extra_gb=os.path.getsize(local) * 0.1 / 1024**3)
    ok, msg = post_write_verify(local, select_bands, band_info, label)
    if not ok:
        log.error('[%s] INTEGRITY FAIL: %s (kept for resume)', label, msg)
        return 'fail'
    log.info('[%s] Integrity OK: %s', label, msg)

    robust_drive_copy(local, final, verify='md5')
    os.remove(local)

    if not KEEP_CHECKPOINTS:
        for fname in os.listdir(out_dir):
            if re.match(rf'^{re.escape(base)}tile{tile["idx"]}_\d+hr\.nc$',
                        fname):
                try:
                    os.remove(os.path.join(out_dir, fname))
                except OSError:
                    pass

    log.info('[%s] EXPORTED -> %s (%.1f MiB)', label, final,
             os.path.getsize(final) / 1024**2)
    return 'ok'


# ════════════════════════════════════════════════════════════════════
#  MAIN PIPELINE
# ════════════════════════════════════════════════════════════════════

def run_pipeline():
    global _run_start
    _run_start = time.time()

    log.info('=' * 62)
    log.info('  Dataset : %s', DATASET_ID)
    log.info('=' * 62)

    DiskWatchdog().start()

    coll_raw = ee.ImageCollection(DATASET_ID)
    n_shards = coll_raw.size().getInfo()

    # FIX(53): monthly mosaic — the core structural fix
    coll_mos = monthly_mosaic(coll_raw)
    n_months = coll_mos.size().getInfo()
    log.info('Collection: %d HUC-shard images -> %d monthly mosaics',
             n_shards, n_months)

    roi = load_roi(SHAPEFILE_DIR, simplify_deg=0.001)
    bands = validate_dataset(coll_mos, roi, select=SELECT_BANDS)
    log.info('Using %d band(s): %s', len(bands), bands)

    # FIX(55): native grid from RAW shards (mosaics report the default
    # 1-degree projection — using it would produce 111 km pixels!)
    grid_crs, nominal_m, native_xform = resolve_native_crs(
        coll_raw.select(bands))

    y0 = YEAR_START or int(ee.Date(
        coll_mos.sort('system:time_start').first()
            .get('system:time_start')).get('year').getInfo())
    y1 = YEAR_END or int(ee.Date(
        coll_mos.sort('system:time_start', False).first()
            .get('system:time_start')).get('year').getInfo())
    years = list(range(y0, y1 + 1))
    log.info('Years : %d-%d (%d yrs, %d blocks of %d yr/tile)',
             y0, y1, len(years), math.ceil(len(years) / BLOCK_YEARS),
             BLOCK_YEARS)

    roi = load_roi(SHAPEFILE_DIR,
                   simplify_deg=(nominal_m * 1.5) / 111_320.0)

    short = DATASET_ID.split('/')[-1]
    base = f'{short}_{ROI_LABEL}_'
    out_dir = os.path.join(DRIVE_ROOT, short)
    os.makedirs(out_dir, exist_ok=True)

    grid_params = build_grid_params(roi, grid_crs, nominal_m,
                                    native_xform)
    tiles = split_tiles(grid_params, TILING)

    auto_types = detect_band_types(coll_raw.select(bands), bands)
    band_info = merge_band_info(bands, BAND_METADATA, auto_types)

    ensure_disk_free('pipeline start')

    results = {'ok': [], 'fail': [], 'skip': [], 'incomplete': []}
    for tile in tiles:
        ensure_disk_free(f'before {tile["label"]}')
        log.info('─' * 62)
        try:
            status = process_tile(tile, grid_crs, nominal_m, band_info,
                                  years, out_dir, base)
        except DataQualityError as e:
            log.error('[%s] DATA ERROR: %s', tile['label'], e)
            status = 'fail'
        results[status].append(tile['label'])
        gc.collect()

    log.info('=' * 62)
    log.info('  TILES  ok=%d  fail=%d  skip=%d  incomplete=%d',
             len(results['ok']), len(results['fail']),
             len(results['skip']), len(results['incomplete']))
    for k, v in results.items():
        if v:
            log.info('    %-10s: %s', k, v)
    log.info('=' * 62)

    for f in sorted(glob.glob(os.path.join(out_dir,
                                           f'{base}tile*.nc'))):
        log.info('  %s  (%.1f MiB)', os.path.basename(f),
                 os.path.getsize(f) / 1024**2)


if __name__ == '__main__':
    run_pipeline()

ERROR:gee_export:[tile0] block error (attempt 1): NetCDF: Attempt to define fill value when data already exists.


In [ ]:
import ee
ee.Initialize(project='msugw-503806',
              opt_url='https://earthengine-highvolume.googleapis.com')
col = ee.ImageCollection("projects/openet/assets/ensemble/conus/gridmet/monthly/v2_1")
img = col.sort('system:time_start').first()
print('n images :', col.size().getInfo())
print('first img:', img.get('system:index').getInfo(), img.date().format().getInfo())
print('BANDS    :', img.bandNames().getInfo())   # <-- the exact names to use

n images : 8986
first img: 10s_19991001_19991031 1999-10-01T00:00:00
BANDS    : ['et_ensemble_mad', 'et_ensemble_mad_min', 'et_ensemble_mad_max', 'et_ensemble_mad_count', 'et_ensemble_mad_index', 'et_ensemble_sam']


In [ ]:
import ee, numpy as np
ee.Initialize(project='msugw-503806',
              opt_url='https://earthengine-highvolume.googleapis.com')

ASSET = "projects/openet/assets/ensemble/conus/gridmet/monthly/v2_1"
coll = ee.ImageCollection(ASSET)

ts = np.array(coll.aggregate_array('system:time_start').getInfo(),
              dtype='datetime64[ms]').astype('datetime64[M]')
u, c = np.unique(ts, return_counts=True)
print('Total images:', len(ts), '| months:', len(u),
      '| images/month min/med/max:', c.min(), int(np.median(c)), c.max())

month = coll.filterDate('2020-06-01', '2020-07-01')
table = (ee.FeatureCollection(month.map(lambda im: ee.Feature(None, {
            'i': ee.String(im.get('system:index')),
            'b': im.bandNames()}))).getInfo())
rows = table['features']
print(f'\n{len(rows)} images in 2020-06:')
for f in rows:
    p = f['properties']
    print(' ', p['i'], '| #bands =', len(p['b']), '|', list(p['b']))

# Actual pixel values at an Ogallala-centre point: first "poor" and first "rich" image
pt = ee.Geometry.Point(-101.5, 38.5)
for f in rows[:2]:
    img = ee.Image(ASSET + '/' + f['properties']['i'])
    print(f['properties']['i'], '-> pixel:',
          img.reduceRegion(ee.Reducer.first(), pt, 30).getInfo())

Total images: 8986 | months: 323 | images/month min/med/max: 14 32 32

32 images in 2020-06:
  10s_20200601_20200630 | #bands = 6 | ['et_ensemble_mad', 'et_ensemble_mad_min', 'et_ensemble_mad_max', 'et_ensemble_mad_count', 'et_ensemble_mad_index', 'et_ensemble_sam']
  10t_20200601_20200630 | #bands = 6 | ['et_ensemble_mad', 'et_ensemble_mad_min', 'et_ensemble_mad_max', 'et_ensemble_mad_count', 'et_ensemble_mad_index', 'et_ensemble_sam']
  10u_20200601_20200630 | #bands = 6 | ['et_ensemble_mad', 'et_ensemble_mad_min', 'et_ensemble_mad_max', 'et_ensemble_mad_count', 'et_ensemble_mad_index', 'et_ensemble_sam']
  11s_20200601_20200630 | #bands = 6 | ['et_ensemble_mad', 'et_ensemble_mad_min', 'et_ensemble_mad_max', 'et_ensemble_mad_count', 'et_ensemble_mad_index', 'et_ensemble_sam']
  11t_20200601_20200630 | #bands = 6 | ['et_ensemble_mad', 'et_ensemble_mad_min', 'et_ensemble_mad_max', 'et_ensemble_mad_count', 'et_ensemble_mad_index', 'et_ensemble_sam']
  11u_20200601_20200630 | #bands = 6 

In [ ]:
import ee
ee.Initialize(project='msugw-503806',
              opt_url='https://earthengine-highvolume.googleapis.com')
ASSET = "OpenET/ETv2_1/CONUS/GRIDMET/MONTHLY/v2_1"
col = ee.ImageCollection(ASSET)

img = col.filterDate('2020-06-01', '2020-07-01').first()
print('bands:', img.bandNames().getInfo())

pt = ee.Geometry.Point(-101.5, 38.5)   # Ogallala interior
print('pixel :', img.select([
    'et_ensemble_mean', 'et_ensemble_median',
    'et_reference_ensemble_mean']).reduceRegion(
        ee.Reducer.first(), pt, 90).getInfo())
# Expect finite numbers like {'et_ensemble_mean': ~100-250 mm, ...}

EEException: ImageCollection.load: ImageCollection asset 'OpenET/ETv2_1/CONUS/GRIDMET/MONTHLY/v2_1' not found (does not exist or caller does not have access).

In [ ]:
import ee
ee.Initialize(project='msugw-503806',
              opt_url='https://earthengine-highvolume.googleapis.com')
ASSET = "projects/openet/assets/ensemble/conus/gridmet/monthly/v2_1"
mos = ee.Image(ee.ImageCollection(ASSET).filterDate('2020-06-01','2020-07-01').mosaic())
print('bands:', mos.bandNames().getInfo())
print('pixel :', mos.reduceRegion(ee.Reducer.first(),
      ee.Geometry.Point(-101.5, 38.5), 30).getInfo())
# Expect real numbers now (et_ensemble_mad ~ 50-250 mm in June)

bands: ['et_ensemble_mad', 'et_ensemble_mad_min', 'et_ensemble_mad_max', 'et_ensemble_mad_count', 'et_ensemble_mad_index', 'et_ensemble_sam']
pixel : {'et_ensemble_mad': 82, 'et_ensemble_mad_count': 4, 'et_ensemble_mad_index': 15, 'et_ensemble_mad_max': 88, 'et_ensemble_mad_min': 71, 'et_ensemble_sam': 83}
